# Environment

In [ ]:
import os
os.chdir("..")

In [ ]:
import pandas as pd
import spacy

from src.utils.path import DATA_PATH

In [ ]:
nlp = spacy.load("pt_core_news_lg")

# Load Data

### Corpora

In [ ]:
lula = pd.read_csv(f"{DATA_PATH}/ited-br/raw/lula.csv")
bolsonaro = pd.read_csv(f"{DATA_PATH}/ited-br/raw/bolsonaro.csv")

### Lexicon

In [ ]:
mol = pd.read_csv(f"{DATA_PATH}/lexicons/mol.csv")

# Filters

## By Relevance (> 10 Tokens)

In [ ]:
def filter_relevant_comments(df, nlp, text_col, min_tokens):
    return df[df[text_col].apply(
        lambda x: len([t for t in nlp(str(x)) if not t.is_stop]) > min_tokens
        )].reset_index(drop=True)

## By Terms

In [ ]:
words = [
    "ladrão", "bandido", "nordeste", "corrupto", "bozo", "corrupção", "preso",
    "nine", "lulinha", "luladrão", "presidiário", "pobre", "crime", "roubar",
    "inocentar", "petista", "esquerda", "mentira", "lulaladrão", "ex presidiario",
    "lulalivre", "9 dedos", "luladrao", "lulaladrao", "cachaceiro", "sapo barbudo",
    "lulao", "l13", "faz o L", "lulindo", "metalurgico", "lulalkimin", "capitão",
    "merda", "genocida", "pior", "pedófilo", "vergonha", "maçonaria", "PT",
    "mito", "clima", "Deus", "mulher", "Roberto", "Jefferson", "Ciro", "mentiu",
    "biroliro", "tchutchuca do centrão", "bonoro", "bolsomito", "bolsolixo",
    "bolsotrump", "messias", "patriota", "b22", "b17", "brocha", "imbrochavel",
    "maçonaro"
]

In [ ]:
filter_terms = set(mol["pt-brazilian-portuguese"].dropna()) | set(words)

In [ ]:
def filter_by_terms(df, terms, text_col, sample=None):
    return df[df[text_col].str.lower().apply(
        lambda x: any(t in x for t in terms)
        )].reset_index(drop=True)

# Processing

## Corpus of Lula

In [ ]:
lula_filtered = filter_relevant_comments(df=lula, nlp=nlp, text_col="text", min_tokens=10)
lula_filtered = filter_by_terms(df=lula_filtered, terms=filter_terms, text_col="text")
lula_filtered = lula_filtered.sample(50000).reset_index(drop=True)

In [ ]:
lula_filtered.info()

In [ ]:
lula_filtered.to_csv(f"{DATA_PATH}/ited-br/raw/lula_50k.csv", index=False)

## Corpus of Bolsonaro

In [ ]:
bolsonaro_filtered = filter_relevant_comments(df=bolsonaro, nlp=nlp, text_col="text", min_tokens=10)
bolsonaro_filtered = filter_by_terms(df=bolsonaro_filtered, terms=filter_terms, text_col="text")
bolsonaro_filtered = bolsonaro_filtered.sample(50000).reset_index(drop=True)

In [ ]:
bolsonaro_filtered.info()

In [ ]:
bolsonaro_filtered.to_csv(f"{DATA_PATH}/ited-br/raw/bolsonaro_50k.csv", index=False)